In [16]:
#Import models and libraries
from DT.DecisionTree import DecisionTree
from sklearn.tree import DecisionTreeClassifier
from GNB.gaussian_naive_bayes import GNB
from LogisticRegresssion.LogisticRegression import LogReg
from tensorflow import keras
from SVM.linear_svm import LinearSVMScartch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report , f1_score
from sklearn.model_selection import train_test_split
from Kfolds import run_kfold
import itertools
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score




In [17]:
#Download Data
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()


In [18]:
# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

In [19]:
import numpy as np
from skimage.feature import hog

def extract_hog_features(X):
    features = []

    for img in X:
        img_2d = img.reshape(28, 28)

        hog_features = hog(
            img_2d,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            feature_vector=True
        )

        features.append(hog_features)

    return np.array(features)

In [20]:
X_train_HOG = extract_hog_features(X_train)
X_test_HOG = extract_hog_features(X_test)

In [21]:
pca = PCA(n_components=50)  

X_train_hog_pca = pca.fit_transform(X_train_HOG)
X_test_hog_pca = pca.transform(X_test_HOG)

In [22]:
print(X_test_hog_pca.shape)

(10000, 50)


In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_hog_pca, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

In [27]:


def manual_grid_search_DT(X, y, param_grid, k=3):

    keys = list(param_grid.keys())
    values = list(param_grid.values())

    best_score = -1
    best_params = None
    results = []

    for combo in itertools.product(*values):

        params = dict(zip(keys, combo))

        print("\nTesting:", params)

        kf = KFold(n_splits=k, shuffle=True, random_state=42)

        fold_scores = []

        for train_idx, val_idx in kf.split(X):

            X_train_fold = X[train_idx]
            X_val_fold = X[val_idx]

            y_train_fold = y[train_idx]
            y_val_fold = y[val_idx]

            model = DecisionTree(
                maxDepth=params["maxDepth"],
                minSamplesSplit=params["minSamplesSplit"],
                minSampleLeafs=params["minSampleLeafs"],
                criterion=params["criterion"],
                maxFeatures=params["maxFeatures"]
            )

            model.fit(X_train_fold, y_train_fold)

            preds = model.predict(X_val_fold)

            score = f1_score(y_val_fold, preds, average="macro")

            fold_scores.append(score)

        avg_score = sum(fold_scores) / len(fold_scores)

        print("Average score:", avg_score)

        results.append((params, avg_score))

        if avg_score > best_score:
            best_score = avg_score
            best_params = params

    print("\nBest params:", best_params)
    print("Best score:", best_score)

    return best_params, best_score, results

In [ ]:
#DT hyper parameter tuning
param_grid = {
    "maxDepth": [8, 12, 15],
    "minSamplesSplit": [5, 10, 20],
    "minSampleLeafs": [1, 5, 10],
    "criterion": ["gini", "entropy"],
    "maxFeatures": ["sqrt", "log2"]
}
best_params, _, _ = manual_grid_search_DT(X_train , y_train , param_grid)




Testing: {'maxDepth': 8, 'minSamplesSplit': 5, 'minSampleLeafs': 1, 'criterion': 'gini', 'maxFeatures': 'sqrt'}


In [ ]:
#Decision Tree results
def train_DT(X, y):
    dt = DecisionTree(**best_params)
    dt.fit(X, y)
    return dt
def predict_DT(model, X):
    return model.predict(X)

#Validation
print("Validation results")
run_kfold(X_val , y_val , train_DT , predict_DT ,k=5, binary=0)

dt = DecisionTree()


dt.fit(X_train , y_train)
predictions = dt.predict(X_test_hog_pca)
print(classification_report(
    y_test, predictions,
))



Validation results
Fold 1 → F1: 0.7169
Fold 2 → F1: 0.7433
Fold 3 → F1: 0.7556


KeyboardInterrupt: 

In [ ]:


# Train
gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

# Tune weights
best_weight = None
best_score = -1

def train_gnb(X, y):
    gnb = GNB()
    gnb.gaussian_naive_train(X, y)
    return gnb


def predict_gnb(model, X):
    return model.predict(
        X)



print("\n=== K-FOLD VALIDATION ===")
run_kfold(X_train, y_train, train_gnb, predict_gnb, k=5 , binary=0)



gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

predictions = gnb.predict(
    X_test_hog_pca, 
)

print(classification_report(
    y_test,
    predictions,
))


=== K-FOLD VALIDATION ===
Fold 1 → F1: 0.9272
Fold 2 → F1: 0.9309
Fold 3 → F1: 0.9293
Fold 4 → F1: 0.9329
Fold 5 → F1: 0.9337

K-Fold Avg F1: 0.9307894835809721
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       980
           1       0.99      0.96      0.97      1135
           2       0.93      0.95      0.94      1032
           3       0.93      0.95      0.94      1010
           4       0.95      0.95      0.95       982
           5       0.95      0.93      0.94       892
           6       0.97      0.94      0.96       958
           7       0.96      0.89      0.92      1028
           8       0.85      0.93      0.89       974
           9       0.90      0.92      0.91      1009

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000

